# Mobile Money Agent Network: Data Clean-Up

**Goal:** turn two messy extracts (transactions and agents) into one trustworthy, analysis-ready table, with every rule applied in code (`cleaning_pipeline.py`) and every change logged.

Flow: generate raw data (seed 44) → audit & clean → validate the merge → summarise by county and by week → Data Quality Report.

In [1]:
import subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd

from cleaning_pipeline import clean_agent_data

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.max_colwidth = 80

## 1. Generate and load the raw data

In [2]:
subprocess.run([sys.executable, "generate_data.py"], check=True)

# Read everything as text so no messiness is hidden by automatic type inference
txns_raw = pd.read_csv("raw/transactions_raw.csv", dtype=str, keep_default_na=False, na_values=[""])
agents_raw = pd.read_csv("raw/agents_raw.csv", dtype=str, keep_default_na=False, na_values=[""])

print(f"Raw transactions: {txns_raw.shape[0]:,} rows x {txns_raw.shape[1]} cols")
print(f"Raw agents:       {agents_raw.shape[0]:,} rows x {agents_raw.shape[1]} cols")
txns_raw.sample(8, random_state=44)

transactions_raw.csv: 2075 rows | agents_raw.csv: 63 rows
Raw transactions: 2,075 rows x 5 cols
Raw agents:       63 rows x 5 cols


,txn_id,agent_id,txn_type,amount,txn_date
996,TX00452,AG021,Withdrawal,1011.49,2026-02-09
494,TX00200,AG050,Deposit,2171.59,2026-02-06
1149,TX00044,AG053,cash-in,4575.32,2026-01-22
324,TX00162,AG044,Deposit,652.38,2026-02-09
431,TX01118,AG036,Withdrawal,"1,456.15","Apr 17, 2026"
1732,TX01003,AG057,Withdrawal,999.90,2026-03-03
2017,TX00762,AG056,Withdrawal,10357.20,2026-03-10
1098,TX01423,AG053,Withdrawal,3411.02,2026-06-05


In [3]:
print("Distinct txn_type spellings in raw data:")
txns_raw["txn_type"].value_counts(dropna=False)

Distinct txn_type spellings in raw data:


txn_type
Deposit         810
Withdrawal      745
DEPOSIT          61
Dep              57
cash-in          52
Withdraw         51
WITHDRAWAL       51
deposit          45
withdrawal       43
 Withdrawal      42
 Deposit         42
cash out         34
w/d              30
NaN              12
Name: count, dtype: int64

## 2. Run the cleaning pipeline

In [4]:
clean_df, log = clean_agent_data(txns_raw, agents_raw)
print(f"clean_agent_data() returned {len(clean_df):,} rows and {len(log)} logged rules")
clean_df.head()

clean_agent_data() returned 1,948 rows and 20 logged rules


,txn_id,agent_id,agent_name,county,tier,txn_type,amount,txn_date,amount_imputed,is_outlier,agent_known
0,TX00093,AG019,Agent 019,Nakuru,Silver,Withdrawal,"4,601.24",2026-01-01,False,False,True
1,TX00138,AG024,Agent 024,Nakuru,Silver,Deposit,"1,297.66",2026-01-01,False,False,True
2,TX01159,AG046,Agent 046,Kakamega,Silver,Deposit,"1,486.58",2026-01-01,False,False,True
3,TX01737,AG055,Agent 055,Mombasa,Bronze,Deposit,"1,449.97",2026-01-01,False,False,True
4,TX01797,AG045,Agent 045,Nairobi,Bronze,Deposit,"2,725.36",2026-01-01,False,False,True


## 3. Data-quality audit table

In [5]:
audit = pd.DataFrame(log, columns=["issue", "dimension", "column", "rows_affected"])
audit

,issue,dimension,column,rows_affected
0,Agents: county text inconsistent (case / padding),Consistency,county,9
1,"Agents: county missing, set to 'Unknown'",Completeness,county,4
2,Agents: duplicate agent rows removed,Uniqueness,agent_id,3
3,"txn_id missing, row removed",Completeness,txn_id,0
4,agent_id typed in lower case / with padding,Consistency,agent_id,31
5,"txn_type spelled inconsistently (case, padding, aliases)",Consistency,txn_type,424
6,Exact duplicate transactions removed,Uniqueness,txn_id,60
7,Repeated txn_id with conflicting values (first record kept),Uniqueness,txn_id,15
8,"txn_type missing, row removed",Completeness,txn_type,12
9,"txn_type not recognised, row removed",Validity,txn_type,0


In [6]:
# Rows affected per dimension (an issue can touch the same row as another issue)
audit.groupby("dimension")["rows_affected"].agg(rules="count", rows_affected="sum").sort_values("rows_affected", ascending=False)

,rules,rows_affected
dimension,,
Consistency,6,1244
Accuracy,1,114
Uniqueness,3,78
Completeness,5,60
Validity,4,32
Timeliness,1,20


## 4. Post-merge validation

In [7]:
# Independent re-computation of the known agent ids (does not reuse pipeline internals)
known_ids = set(agents_raw["agent_id"].str.strip().str.upper())
unknown_check = (~clean_df["agent_id"].isin(known_ids)).sum()

validation = pd.DataFrame({
    "check": [
        "Raw transactions rows",
        "Raw agents rows",
        "Unique agents after de-duplication",
        "Clean (merged) transactions rows",
        "Rows removed from transactions",
        "Duplicate txn_id in clean data",
        "Transactions with unknown agents (pipeline flag)",
        "Transactions with unknown agents (independent check)",
        "Share of transactions with unknown agents",
    ],
    "value": [
        len(txns_raw), len(agents_raw), len(known_ids), len(clean_df),
        len(txns_raw) - len(clean_df), int(clean_df["txn_id"].duplicated().sum()),
        int((~clean_df["agent_known"]).sum()), int(unknown_check),
        f"{(~clean_df['agent_known']).mean():.1%}",
    ],
})
assert clean_df["txn_id"].is_unique, "duplicate txn_id after merge"
assert (~clean_df["agent_known"]).sum() == unknown_check, "unknown-agent counts disagree"
assert clean_df["agent_id"].isin(known_ids).eq(clean_df["agent_known"]).all()
print("All post-merge assertions passed: merge did not multiply rows and unknown agents reconcile.")
validation

All post-merge assertions passed: merge did not multiply rows and unknown agents reconcile.


,check,value
0,Raw transactions rows,2075
1,Raw agents rows,63
2,Unique agents after de-duplication,60
3,Clean (merged) transactions rows,1948
4,Rows removed from transactions,127
5,Duplicate txn_id in clean data,0
6,Transactions with unknown agents (pipeline flag),38
7,Transactions with unknown agents (independent check),38
8,Share of transactions with unknown agents,2.0%


In [8]:
# County assigned to unknown-agent transactions, and rows by merge status
clean_df.groupby("agent_known").agg(rows=("txn_id", "count"), counties=("county", lambda s: ", ".join(sorted(s.unique())[:3]) + ("..." if s.nunique() > 3 else "")))

,rows,counties
agent_known,,
False,38,Unknown
True,1910,"Kakamega, Kiambu, Kisumu..."


## 5. Save the clean dataset and verify the output file

In [9]:
Path("outputs").mkdir(exist_ok=True)
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)

out = pd.read_csv("outputs/agent_transactions_clean.csv")
assert out["txn_id"].is_unique
assert set(out["txn_type"].unique()) == {"Deposit", "Withdrawal"}
assert pd.api.types.is_numeric_dtype(out["amount"]) and out["amount"].notna().all()
assert out["txn_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
assert {"amount_imputed", "is_outlier", "county"}.issubset(out.columns)
print(f"outputs/agent_transactions_clean.csv verified: {len(out):,} rows, columns = {list(out.columns)}")

outputs/agent_transactions_clean.csv verified: 1,948 rows, columns = ['txn_id', 'agent_id', 'agent_name', 'county', 'tier', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'agent_known']


## 6. County summary

In [10]:
county_summary = (
    out.groupby("county")
       .agg(transactions=("txn_id", "count"),
            agents=("agent_id", "nunique"),
            total_value_kes=("amount", "sum"),
            avg_txn_kes=("amount", "mean"),
            deposits=("txn_type", lambda s: (s == "Deposit").sum()),
            withdrawals=("txn_type", lambda s: (s == "Withdrawal").sum()),
            outliers=("is_outlier", "sum"),
            imputed_amounts=("amount_imputed", "sum"))
       .sort_values("total_value_kes", ascending=False)
)
county_summary.loc["TOTAL"] = [county_summary["transactions"].sum(), out["agent_id"].nunique(),
                               county_summary["total_value_kes"].sum(), out["amount"].mean(),
                               county_summary["deposits"].sum(), county_summary["withdrawals"].sum(),
                               county_summary["outliers"].sum(), county_summary["imputed_amounts"].sum()]
county_summary

,transactions,agents,total_value_kes,avg_txn_kes,deposits,withdrawals,outliers,imputed_amounts
county,,,,,,,,
Nairobi,561.00,17.00,"2,317,009.58","4,130.14",272.00,289.00,30.00,16.00
Kakamega,219.00,7.00,"1,499,585.05","6,847.42",116.00,103.00,18.00,3.00
Kiambu,261.00,8.00,"1,241,432.87","4,756.45",136.00,125.00,16.00,8.00
Machakos,181.00,6.00,"940,776.08","5,197.66",90.00,91.00,11.00,6.00
Mombasa,245.00,8.00,"849,006.65","3,465.33",130.00,115.00,12.00,5.00
Nakuru,123.00,4.00,"567,219.38","4,611.54",67.00,56.00,9.00,6.00
Unknown,171.00,9.00,"559,165.55","3,269.97",91.00,80.00,7.00,5.00
Uasin Gishu,125.00,4.00,"403,709.44","3,229.68",65.00,60.00,5.00,6.00
Kisumu,62.00,2.00,"210,640.58","3,397.43",41.00,21.00,6.00,1.00


## 7. Weekly pivot table (value in KES, weeks start on Monday)

In [11]:
out["txn_date"] = pd.to_datetime(out["txn_date"])
out["week_start"] = (out["txn_date"] - pd.to_timedelta(out["txn_date"].dt.weekday, unit="D")).dt.strftime("%Y-%m-%d")

weekly_pivot = out.pivot_table(index="week_start", columns="txn_type", values="amount",
                               aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
weekly_pivot["Net flow (Deposits - Withdrawals)"] = weekly_pivot["Deposit"] - weekly_pivot["Withdrawal"]
weekly_pivot

txn_type,Deposit,Withdrawal,Total,Net flow (Deposits - Withdrawals)
week_start,,,,
2025-12-29,"73,000.16","45,448.87","118,449.03","27,551.29"
2026-01-05,"329,435.22","113,308.28","442,743.50","216,126.94"
2026-01-12,"170,896.06","128,849.72","299,745.78","42,046.34"
2026-01-19,"105,540.80","148,219.58","253,760.38","-42,678.78"
2026-01-26,"811,637.42","127,388.43","939,025.85","684,248.99"
2026-02-02,"134,006.31","306,385.10","440,391.41","-172,378.79"
2026-02-09,"129,057.61","115,063.15","244,120.76","13,994.46"
2026-02-16,"116,589.06","339,666.28","456,255.34","-223,077.22"
2026-02-23,"179,459.68","141,961.14","321,420.82","37,498.54"


In [12]:
# Same pivot by number of transactions
out.pivot_table(index="week_start", columns="txn_type", values="txn_id", aggfunc="count",
                fill_value=0, margins=True, margins_name="Total")

txn_type,Deposit,Withdrawal,Total
week_start,,,
2025-12-29,23,17,40
2026-01-05,53,37,90
2026-01-12,48,42,90
2026-01-19,37,48,85
2026-01-26,35,42,77
2026-02-02,44,34,78
2026-02-09,40,38,78
2026-02-16,35,39,74
2026-02-23,39,44,83


# Data Quality Report

**Scope:** 2,075 raw transaction rows and 63 raw agent rows (seed 44) → **1,948** clean transactions in `outputs/agent_transactions_clean.csv` (127 rows removed, 6.1%). Every figure below comes from the audit table above and is reproducible by re-running the notebook.

### 1. Completeness
- **12** transactions had no `txn_type` and **20** had no `txn_date`; they cannot be classified or placed in a week, so they were **removed**.
- **24** missing `amount` values were **imputed** with the median for the same `txn_type` and flagged `amount_imputed = True`.
- **4** agents had no county, so their transactions carry `county = "Unknown"`. No `txn_id` was missing.

### 2. Validity
- **19** amounts were non-numeric text (`abc`, `N/A`, `?`) and **13** were zero or negative; all were set to missing, imputed as above and flagged.
- The date parser and the `txn_type` mapper both include a drop rule for unrecognised values; in this extract it affected **0** rows.

### 3. Uniqueness
- **60** exact duplicate transactions removed.
- **15** `txn_id`s appeared again with a *different* amount; the **first** record was kept. This is a judgement call, and source-system timestamps would be needed to pick the correct version.
- **3** duplicate agent rows removed, so the merge cannot multiply transactions.

### 4. Consistency
- `txn_type`: **424** rows used aliases, mixed case or padding (`dep`, `w/d`, `cash out`, ` Deposit `) → standardised to only *Deposit* / *Withdrawal*.
- `txn_date`: **547** rows were in `dd/mm/yyyy` or `Mon dd, yyyy` → converted to ISO `YYYY-MM-DD`.
- `amount`: **195** values had `KES` prefixes or thousands separators → converted to numeric.
- `agent_id`: **31** lower-case / padded ids and **9** agent county spellings normalised.
- **Referential integrity:** **38** transactions (2.0%) reference an agent not in the agents table. They were **kept** (the money moved) with `agent_known = False` and `county = "Unknown"`.

### 5. Accuracy
- **114** transactions (5.9%) are statistical outliers (outside 1.5 × IQR within their `txn_type`). They were **flagged, not removed**: some are genuine large transactions, but a cluster of extreme values (30-60× normal) looks like keying or system errors and needs business review.
- **56** amounts (2.9%) are estimates from imputation. Totals by county or week include them, so filter on `amount_imputed` for audited figures.

### 6. Timeliness
- Valid reporting window: **2026-01-01 to 2026-06-30**. **20** rows fell outside it (**12** future-dated 2027 entries and **8** stale 2019 entries) and were **removed**.

### Recommendations
1. Enforce a fixed `txn_type` list, numeric `amount` and ISO dates at the point of capture.
2. Reject or quarantine transactions whose `agent_id` is not registered, and review the 38 orphan cases with the agent-onboarding team.
3. Add a created-at timestamp so conflicting duplicates can be resolved by the latest record.
4. Review flagged outliers before using totals for commission or fraud decisions.
